In [ ]:
import pandas as pd

In [ ]:
import numpy as np

In [ ]:
import itertools

In [ ]:
from collections import Counter

In [ ]:
deployments = pd.read_csv("../data/raw/deployments.csv")

In [ ]:
proposals = pd.read_csv(
    "../data/raw/proposals.csv",
    low_memory=False # Disable low memory mode to avoid DtypeWarning
) # If this fails, almost certainly the CSV files aren't in data/raw/

In [ ]:
deployments["platform"].value_counts() # How many DAOs per platform?

In [ ]:
proposals["platform"].value_counts() # How many proposals per platform?

In [ ]:
deployments["deployment_id"].duplicated().sum() # Is there any duplicate deployment_id?

In [ ]:
pd.crosstab(
    deployments["platform"],
    deployments["estimated_vp"].isna()
) # For each platform, how many deployments have missing 'estimated_vp' values?

In [ ]:
np.isinf(deployments["estimated_vp"]).sum() # From the remaining deployments, how many have infinite 'estimated_vp' values?

In [ ]:
(deployments["estimated_vp"] == 0).sum() # From the remaining deployments, how many have a value of 0 for 'estimated_vp'?

In [ ]:
pd.crosstab(
    proposals["deployment_id"].isna(),
    proposals["platform_deployment_id"].isna()
) # Is 'platform_deployment_id' filled exactly when 'deployment_id' is missing?
# Zeros on the diagonal mean the two columns never overlap: they are complementary.

In [ ]:
missing_id = proposals[proposals["deployment_id"].isna()] # Keep only the proposals without deployment_id

In [ ]:
missing_id.shape

In [ ]:
missing_id["platform"].value_counts() # Which platforms are missing 'deployment_id' in the proposals dataset?

In [ ]:
missing_id["platform_deployment_id"].notna().sum() # Do all of them have the fallback identifier?

In [ ]:
missing_id["platform_deployment_id"].isin(deployments["deployment_id"]).sum() # Are the fallback values also present in the deployments dataset?

In [ ]:
missing_id["platform_deployment_id"].head() # Show the first few fallback values

In [ ]:
deployments[deployments["platform"] == "snapshot"]["deployment_id"].head() # Compare with Snapshot ids in deployments: same *.eth format

In [ ]:
proposals[proposals["total_vp"].isna()]["platform"].value_counts() # Which platforms have missing 'total_vp' values? (same 20,112 Snapshot rows)

In [ ]:
# Create a new column 'dao_id': 'deployment_id' where available, 'platform_deployment_id' where it is missing.
proposals["dao_id"] = proposals["deployment_id"].fillna(
    proposals["platform_deployment_id"]
) # With this column every proposal has an identifier for the DAO it belongs to.

proposals["dao_id"].isna().sum() # Should be 0

In [ ]:
# Proposals whose 'deployment_id' is not found in the deployments dataset.
# At first sight they look like "orphan" proposals, but see the next cells.
orphan_proposals = proposals[
    proposals["deployment_id"].notna()
    & ~proposals["deployment_id"].isin(deployments["deployment_id"])
]

In [ ]:
orphan_proposals.shape # Number of apparent orphan proposals

In [ ]:
orphan_proposals["platform"].value_counts() # Apparent orphans by platform

In [ ]:
deployments[deployments["platform"] == "tally"]["deployment_id"].head() # Tally ids in deployments.csv: all lowercase

In [ ]:
proposals[proposals["platform"] == "tally"]["deployment_id"].head() # Tally ids in proposals.csv: mixed case (checksum format)

In [ ]:
# Ethereum addresses are case-insensitive, so the same DAO is written in two ways.
# Compare both sides in lowercase to see how many apparent orphans are just this.
deployment_ids_lower = set(deployments["deployment_id"].str.lower())
recovered = orphan_proposals["deployment_id"].str.lower().isin(deployment_ids_lower)

In [ ]:
recovered.sum().item() # type: ignore  # Apparent orphans that match once lowercased (all the Tally ones)

In [ ]:
orphan_proposals[~recovered]["platform"].value_counts() # Real orphans: DAO genuinely absent from deployments.csv

In [ ]:
proposals[proposals["author"].isna()]["platform"].value_counts() # Which platforms have missing 'author' values? (all of Tally and Realms)

In [ ]:
proposals["author"].nunique() # Number of distinct authors

In [ ]:
# Identify duplicated proposal_id (keep=False marks every repetition, not only the second one)
duplicated_proposals = proposals[
    proposals["proposal_id"].duplicated(keep=False)
]

In [ ]:
duplicated_proposals.shape

In [ ]:
duplicated_proposals["platform"].value_counts() # All from Tally

In [ ]:
proposals.duplicated().sum() # Rows identical in every column

In [ ]:
# Describe the statistics for the deployment metrics (the 50% row is the median)
deployments[["proposals_count", "unique_voters", "votes_count"]].describe()

In [ ]:
# Count deployments with no unique voters
(deployments["unique_voters"] == 0).sum().item() # type: ignore

In [ ]:
# Count deployments with at least 10 unique voters
(deployments["unique_voters"] >= 10).sum().item() # type: ignore

In [ ]:
# Count deployments with at least 100 unique voters
(deployments["unique_voters"] >= 100).sum().item() # type: ignore

In [ ]:
dates = pd.to_datetime(proposals["date"], errors="coerce") # Text to date; unreadable values become NaT instead of raising an error

In [ ]:
dates.min(), dates.max() # Date range covered by the proposals

In [ ]:
# Count the number of missing dates in the proposals dataset
dates.isna().sum().item() # type: ignore

In [ ]:
linked = proposals.dropna(subset=["dao_id", "author"]) # Drop proposals with missing 'dao_id' or 'author'

In [ ]:
daos_per_author = linked.groupby("author")["dao_id"].nunique() # Count the number of distinct DAOs each author has submitted proposals to

In [ ]:
# Number of authors who have submitted proposals to at least 2 distinct DAOs
# Only these authors can create a connection between two DAOs.
(daos_per_author >= 2).sum().item() # type: ignore

In [ ]:
multi_dao_authors = daos_per_author[daos_per_author >= 2].index # Get the authors who have submitted proposals to at least 2 distinct DAOs

In [ ]:
multi = linked[linked["author"].isin(multi_dao_authors)] # Keep only the proposals from those authors

In [ ]:
# Build the edges: for each author, every pair of DAOs they proposed in is one connection.
# sorted() makes A-B and B-A the same pair. The counter adds 1 per shared author,
# so the final value of each pair is the edge weight.
edges = Counter()

for daos in multi.groupby("author")["dao_id"].unique():
    for a, b in itertools.combinations(sorted(daos), 2):
        edges[(a, b)] += 1

len(edges) # Number of edges

In [ ]:
nodes = set(itertools.chain.from_iterable(edges)) # Flatten all pairs into one list of DAOs and remove repetitions

In [ ]:
len(nodes) # Number of DAOs with at least one connection